In [1]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams['animation.embed_limit'] = 60.0


num_leaves = 79 
N = num_leaves + 1 
G = nx.star_graph(num_leaves)
pos = nx.spring_layout(G, k=0.3, seed=42) # k adjusted for better spacing

K = 2.5    
Dt = 0.05  
steps = 300 

def initialize_star_80():
    global theta, omega
    np.random.seed(42)
    theta = np.random.uniform(0, 2*np.pi, N)
    omega = np.zeros(N)
    for i in range(N):
        if i == 0:
            omega[i] = 1.5  # The Hub
        else:
            omega[i] = 1.0 + np.random.uniform(-0.5, 0.5)


initialize_star_80()


import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


N = 80
k = 4
K = 10.0
Dt = 0.05
p_values = [0.0, 0.2, 1.0] 


graphs = [nx.watts_strogatz_graph(N, k, p, seed=42) for p in p_values]
thetas = [np.random.uniform(0, 2*np.pi, N) for _ in p_values]
omega = 2.0 + np.random.uniform(-0.03, 0.03, N)
pos = {i: (np.cos(2*np.pi*i/N), np.sin(2*np.pi*i/N)) for i in range(N)}


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
scatters = []

for i, ax in enumerate(axes):
    ax.set_aspect('equal')
    ax.axis('off')
    nx.draw_networkx_edges(graphs[i], pos, ax=ax, alpha=0.1, edge_color='gray')
    s = ax.scatter([pos[j][0] for j in range(N)], [pos[j][1] for j in range(N)], 
                   c=thetas[i], cmap='hsv', s=60, edgecolors='black', vmin=0, vmax=2*np.pi)
    scatters.append(s)


def update(frame):
    for i in range(len(p_values)):
        G = graphs[i]
        theta = thetas[i]
        
        
        d_theta = np.zeros(N)
        for node in G.nodes():
            neighbors = list(G.neighbors(node))
            coupling = sum(np.sin(theta[j] - theta[node]) for j in neighbors) / k
            d_theta[node] = omega[node] + (K * coupling)
        
        thetas[i] += d_theta * Dt
        scatters[i].set_array(thetas[i] % (2*np.pi))
        r = np.abs(np.mean(np.exp(1j * thetas[i])))
        axes[i].set_title(f"p={p_values[i]} | r={r:.2f}")
    return scatters


ani = FuncAnimation(fig, update, frames=200, interval=50, blit=True)
plt.close() 

HTML(ani.to_html5_video())